# Baseline models

Baselines for 3-class sentiment (negative / neutral / positive) on Financial PhraseBank.
All models are trained on `train.csv` and evaluated on `val.csv`.

Main metric: **macro F1**, because the classes are imbalanced (59% neutral, 12% negative). A model that never predicts negative can still get a good accuracy, but not a good macro F1.

Models in this notebook:
1. TF-IDF + logistic regression
2. LSTM with random embeddings
3. CNN with random embeddings (frozen and fine-tuned)

The CNN with FinBERT embeddings is in notebook 02 and the FinBERT transformer in notebook 05.

### Load required packages

In [ ]:
import os
import re

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import Input, Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Conv1D, GlobalMaxPooling1D
from keras.optimizers import Adam
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, Callback
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

# FinBERT is downloaded into the project folder
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
from transformers import BertModel, BertTokenizerFast

# Fix the random seed
keras.utils.set_random_seed(42)

### Loading the dataset

In [ ]:
train = pd.read_csv("../data/train.csv")
val = pd.read_csv("../data/val.csv")
print(train.shape, val.shape)
train.head()

In [ ]:
train.label.value_counts(normalize=True)

In [ ]:
# encode the labels as negative = 0, neutral = 1, positive = 2
LABELS = ["negative", "neutral", "positive"]
y_train = train.label.map({"negative": 0, "neutral": 1, "positive": 2}).values
y_val = val.label.map({"negative": 0, "neutral": 1, "positive": 2}).values

In [ ]:
lengths = [len(s.split()) for s in train.sentence]
print("Longest sentence:", max(lengths), "words")
print("Median length:", int(np.median(lengths)), "words")

### Preprocessing

We lowercase every sentence and replace every number with `<num>`. The sentiment depends on the structure ("profit rose to <num>") and not on the exact number (Symeonidis et al., 2018). This also makes the vocabulary smaller (7,840 to 7,081 words).

We do not remove stopwords: lists like NLTK's contain words such as *not*, *up* and *down*, which carry the sentiment in financial text.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+([.,]\d+)*", " <num> ", text)   # 2.5 or 1,000 -> <num>
    text = re.sub(r"\s+", " ", text).strip()
    return text

print(train.sentence[0])
print(clean_text(train.sentence[0]))

### Function to evaluate a model

Prints accuracy, macro F1 and the report per class, and saves the confusion matrix.

In [ ]:
def evaluate(probs, name):
    pred = probs.argmax(axis=1)
    print("accuracy:", round(accuracy_score(y_val, pred), 4))
    print("macro F1:", round(f1_score(y_val, pred, average="macro"), 4))
    print(classification_report(y_val, pred, target_names=LABELS))
    disp = ConfusionMatrixDisplay.from_predictions(y_val, pred, display_labels=LABELS, cmap="Reds")
    disp.ax_.set_title(name + ": confusion matrix (val)")
    plt.savefig("../results/figures/" + name.lower().replace(" ", "_").replace(",", "") + "_cm.png", dpi=150)
    plt.show()

## 1. TF-IDF + logistic regression

Unigrams and bigrams, so phrases like "went down" are one feature. The vectorizer is fitted on train only.

In [ ]:
vectorizer = TfidfVectorizer(preprocessor=clean_text, tokenizer=str.split, token_pattern=None,
                             ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(train.sentence)
X_val_tfidf = vectorizer.transform(val.sentence)
print(X_train_tfidf.shape)

In [ ]:
logreg = LogisticRegression(C=10, class_weight="balanced", max_iter=2000)
logreg.fit(X_train_tfidf, y_train)

In [ ]:
probs_logreg = logreg.predict_proba(X_val_tfidf)
evaluate(probs_logreg, "Baseline logreg")

## 2. LSTM

The Keras tokenizer turns every word into an integer (fitted on train only). Sentences are padded / cut to 64 tokens.

In [ ]:
filters = '!"#$&()*+,-./:;=?@[\\]^_`{|}~\t\n'   # Keras default without < and >, so <num> stays one token
tokenizer = Tokenizer(oov_token="<OOV>", filters=filters)
tokenizer.fit_on_texts(train.sentence.apply(clean_text))

X_train = pad_sequences(tokenizer.texts_to_sequences(train.sentence.apply(clean_text)), maxlen=64, padding="post")
X_val = pad_sequences(tokenizer.texts_to_sequences(val.sentence.apply(clean_text)), maxlen=64, padding="post")
vocab_size = len(tokenizer.word_index) + 1
print(X_train.shape, X_val.shape, vocab_size)

In [ ]:
i = 0
print(train.sentence[i])
print(X_train[i])

a. Define the model and print its summary

In [ ]:
model_lstm = Sequential()
model_lstm.add(Input(shape=(64,)))
model_lstm.add(Embedding(vocab_size, 100, mask_zero=True))
model_lstm.add(LSTM(64))
model_lstm.add(Dropout(0.3))
model_lstm.add(Dense(3, activation="softmax"))
model_lstm.summary()

b. Compile the model

In [ ]:
model_lstm.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

c. Train the model (stops when the val loss has not improved for 3 epochs)

In [ ]:
history_lstm = model_lstm.fit(X_train, to_categorical(y_train, 3),
                              validation_data=(X_val, to_categorical(y_val, 3)),
                              epochs=30, batch_size=32,
                              callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])

In [ ]:
# visualise training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history_lstm.history["loss"], label="train")
ax1.plot(history_lstm.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.set_xlabel("epoch")
ax1.legend()
ax2.plot(history_lstm.history["accuracy"], label="train")
ax2.plot(history_lstm.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.legend()
plt.tight_layout()
plt.savefig("../results/figures/baseline_lstm_loss.png", dpi=150)
plt.show()

In [ ]:
probs_lstm = model_lstm.predict(X_val, verbose=0)
evaluate(probs_lstm, "Baseline LSTM")

Logistic regression is better than the LSTM. The most likely reason is the data size: the LSTM has ~750k parameters and only 2,901 training sentences, so it overfits after ~2 epochs. The sentences are also short (median 21 words), so modelling word order does not help much.

## 3. CNN with random embeddings

Same CNN as the FinBERT CNN in notebook 02, but the embedding matrix is **random numbers** with the same shape and scale as FinBERT's.
The only thing missing is the pretrained knowledge, so the gap to the FinBERT CNN shows how much the pretrained embeddings add.

- **random frozen:** the random vectors never change (the floor)
- **random fine-tuned:** the embeddings are learned from the 2,901 training sentences only

The CNN uses FinBERT's word pieces as tokens (not the LSTM tokenizer above), so it is the same input as in notebook 02.

In [ ]:
bert_tokenizer = BertTokenizerFast.from_pretrained("yiyanghkust/finbert-pretrain")
X_train_wp = bert_tokenizer(list(train.sentence), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_val_wp = bert_tokenizer(list(val.sentence), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]

# we only need FinBERT's embedding matrix for its shape and scale
finbert = BertModel.from_pretrained("yiyanghkust/finbert-pretrain")
finbert_matrix = finbert.embeddings.word_embeddings.weight.detach().numpy()
del finbert

rng = np.random.default_rng(42)
random_matrix = rng.normal(0, finbert_matrix.std(), finbert_matrix.shape).astype("float32")
print(X_train_wp.shape, random_matrix.shape)

After every epoch we compute macro F1 on val, so early stopping can use it (Keras only has accuracy and loss built in).

In [ ]:
class MacroF1(Callback):
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(X_val_wp, batch_size=32, verbose=0).argmax(axis=1)
        logs["val_macro_f1"] = f1_score(y_val, pred, average="macro")

In [ ]:
def build_cnn(embedding_matrix, trainable):
    model = Sequential()
    model.add(Input(shape=(64,)))
    # start from zeros and copy the matrix in below (a Constant initializer uses too much memory)
    model.add(Embedding(embedding_matrix.shape[0], embedding_matrix.shape[1], embeddings_initializer="zeros", trainable=trainable))
    model.add(SpatialDropout1D(0.3))
    model.add(Conv1D(64, 3, activation="relu"))
    model.add(GlobalMaxPooling1D())
    model.add(Dense(32, activation="relu"))
    model.add(Dropout(0.5))
    model.add(Dense(3, activation="softmax"))
    model.layers[0].set_weights([embedding_matrix])
    model.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

### a. Random frozen

In [ ]:
keras.utils.set_random_seed(42)
cnn_frozen = build_cnn(random_matrix, trainable=False)
cnn_frozen.summary()

In [ ]:
early_stop = EarlyStopping(monitor="val_macro_f1", mode="max", patience=5, restore_best_weights=True)
history_frozen = cnn_frozen.fit(X_train_wp, to_categorical(y_train, 3),
                                validation_data=(X_val_wp, to_categorical(y_val, 3)),
                                epochs=20, batch_size=32, verbose=0, callbacks=[MacroF1(), early_stop])

In [ ]:
probs_random_frozen = cnn_frozen.predict(X_val_wp, batch_size=32, verbose=0)
evaluate(probs_random_frozen, "Baseline CNN random frozen")

### b. Random fine-tuned

In [ ]:
keras.utils.set_random_seed(42)
cnn_finetuned = build_cnn(random_matrix, trainable=True)
cnn_finetuned.summary()

In [ ]:
early_stop = EarlyStopping(monitor="val_macro_f1", mode="max", patience=5, restore_best_weights=True)
history_finetuned = cnn_finetuned.fit(X_train_wp, to_categorical(y_train, 3),
                                      validation_data=(X_val_wp, to_categorical(y_val, 3)),
                                      epochs=20, batch_size=32, verbose=0, callbacks=[MacroF1(), early_stop])

In [ ]:
probs_random_finetuned = cnn_finetuned.predict(X_val_wp, batch_size=32, verbose=0)
evaluate(probs_random_finetuned, "Baseline CNN random finetuned")

In [ ]:
# visualise the val macro F1 of both CNNs
plt.plot(history_frozen.history["val_macro_f1"], label="random frozen")
plt.plot(history_finetuned.history["val_macro_f1"], label="random fine-tuned")
plt.title("CNN with random embeddings")
plt.xlabel("epoch")
plt.ylabel("val macro F1")
plt.legend()
plt.show()

## Save the results

One row per model in `results/metrics.csv` (a row of the same model is replaced, so running again does not add duplicates).
`hard` = the val sentences in the 0.4-0.6 confidence band of the FinBERT CNN (`results/hard_samples/band_val.csv`, notebook 02), the same for every model.

In [ ]:
hard = pd.read_csv("../results/hard_samples/band_val.csv").is_hard.values

def metrics_row(probs, embedding_config, architecture):
    pred = probs.argmax(axis=1)
    per_class = f1_score(y_val, pred, labels=[0, 1, 2], average=None)
    return {"condition": "baseline", "embedding_config": embedding_config, "architecture": architecture,
            "accuracy_overall": accuracy_score(y_val, pred),
            "f1_overall": f1_score(y_val, pred, average="macro"),
            "accuracy_hard": accuracy_score(y_val[hard], pred[hard]),
            "f1_hard": f1_score(y_val[hard], pred[hard], average="macro"),
            "accuracy_nonhard": accuracy_score(y_val[~hard], pred[~hard]),
            "f1_nonhard": f1_score(y_val[~hard], pred[~hard], average="macro"),
            "f1_negative": per_class[0], "f1_neutral": per_class[1], "f1_positive": per_class[2]}

rows = [metrics_row(probs_logreg, "tfidf_1-2gram", "logreg"),
        metrics_row(probs_lstm, "random_init_100d", "lstm_64_dropout"),
        metrics_row(probs_random_frozen, "random_frozen", "cnn_64"),
        metrics_row(probs_random_finetuned, "random_finetuned", "cnn_64")]

metrics = pd.read_csv("../results/metrics.csv")
metrics = pd.concat([metrics, pd.DataFrame(rows)], ignore_index=True)
metrics = metrics.drop_duplicates(subset=["condition", "embedding_config", "architecture"], keep="last")
metrics.to_csv("../results/metrics.csv", index=False)
pd.DataFrame(rows)[["embedding_config", "architecture", "accuracy_overall", "f1_overall"]].round(3)

We also save the val probabilities of every model, for the error analysis later.

In [ ]:
os.makedirs("../results/predictions", exist_ok=True)

def save_probs(probs, name):
    out = val[["sentence", "label"]].copy()
    out["p_negative"] = probs[:, 0]
    out["p_neutral"] = probs[:, 1]
    out["p_positive"] = probs[:, 2]
    out["p_true"] = probs[np.arange(len(val)), y_val]
    out["pred"] = [LABELS[i] for i in probs.argmax(axis=1)]
    out.to_csv("../results/predictions/" + name + ".csv", index=False)

save_probs(probs_logreg, "baseline_logreg")
save_probs(probs_lstm, "baseline_lstm")
save_probs(probs_random_frozen, "baseline_cnn_random_frozen")
save_probs(probs_random_finetuned, "baseline_cnn_random_finetuned")